In [ ]:
from pathlib import Path
from typing import List, Optional, Tuple
import pickle
from itertools import chain

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchdiffeq
from joblib import Parallel, delayed
from lightning.pytorch import (
    seed_everything,
)
from seaborn.algorithms import bootstrap
from scipy.stats.contingency import association
from scipy.stats import pearsonr
from torch import Tensor

from patientflow.data import BrainteaserDataModule
from patientflow.models.ae import PatientEmbeddedAE
from patientflow.models.vector_fields import MLP, TransformerVecField

In [ ]:
SEED = 42
seed_everything(SEED)

In [ ]:
ds = BrainteaserDataModule(
    SEED,
    32,
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year",
)
ds.setup("fit")

In [ ]:
x_s = ds.train_dataset[:][0]
x_t = ds.train_dataset[:][1]
real_seq_sizes = ds.train_dataset[:][2]

In [ ]:
ae: PatientEmbeddedAE = torch.load("../vae_2025-03-27 15:13:04.pt").eval()

In [ ]:
with torch.no_grad():
    encoded_static, encoded_temporal = ae.encode(x_s, x_t)
    if isinstance(encoded_static, tuple):
        encoded_static = encoded_static[0]
    if isinstance(encoded_temporal, tuple):
        encoded_temporal = encoded_temporal[0]
    recons_static, recons_temporal = ae.decode(
        encoded_static,
        encoded_temporal,
        transform_and_unify_output=True,
        adjust_to_seq_lengths=real_seq_sizes,
    )

In [ ]:
version = "2025-03-28 15:06:33"
device = torch.device("cuda")
net_static = torch.load(f"../net_static_{version}.pt").to(device).eval()

In [ ]:
net_temporal = torch.load(f"../net_temporal_{version}.pt").to(device).eval()

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)

real = pd.concat(dfs, ignore_index=True)

In [ ]:
numerical_mosaic_grid = [["Age_onset", "Height (m)", "Weight"]]

categorical_mosaic_grid = [
    ["Gender", "Ethnicity", "NIV", "Tracheostomy"],
    ["PEG", "UMNvsLMN", "Onset", "Limb_O"],
    ["Limbs_Impairment", "Limbs_Side", "Weightloss_>10%", "ALS_familiar_history"],
    ["Ever_smoked", "Blood_hypertension", "Diabetes", "Dyslipidemia"],
    ["Thyroid", "Autoimmune", "SOD1 Mutation", "Stroke"],
    ["Cardiac_disease", "Primary_cancer", "C9orf72", "TARDBP mutation"],
    ["FUS mutation", ".", ".", "."],
]

categorical_feats = list(chain.from_iterable(categorical_mosaic_grid))
numerical_feats = list(chain.from_iterable(numerical_mosaic_grid))
categorical_inverted_x_axis_feats = [
    "Ethnicity",
    "Onset",
    "Limb_O",
    "Limbs_Impairment",
    "Limbs_Side",
    "SOD1 Mutation",
    "TARDBP mutation",
    "FUS mutation",
]

In [ ]:
N_SAMPLES = 100

categorical_feats_info = {}

gen_dfs = []
GENERATE = True
SAMPLING_GAMMA = 2.0

In [ ]:
def generate_dataset(num_ode_steps: int) -> Tuple[Tensor, Tensor]:
    with torch.no_grad():
        seq_sizes = torch.multinomial(
            ds.seq_lengths_weights, encoded_static.size(0), replacement=True
        )
        ode_times = torch.linspace(0, 1, num_ode_steps, device=device)

        traj_static = torchdiffeq.odeint(
            lambda t, x: net_static(torch.concat((x, t.repeat(x.size(0), 1)), dim=1)),
            torch.randn(encoded_static.size(0), encoded_static.size(1), device=device),
            ode_times,
            atol=1e-4,
            rtol=1e-4,
            method="dopri5",
        )

        traj_temporal = torchdiffeq.odeint(
            lambda t, x: SAMPLING_GAMMA
            * net_temporal(t, x, traj_static[-1], seq_sizes=seq_sizes.to(device))
            + (1 - SAMPLING_GAMMA)
            * net_temporal(
                t,
                x,
                torch.zeros_like(traj_static[-1]).to(device),
                seq_sizes=seq_sizes.to(device),
            ),
            torch.randn(
                encoded_temporal.size(0),
                31,
                encoded_temporal.size(2),
                device=device,
            ),
            ode_times,
            atol=1e-4,
            rtol=1e-4,
            method="dopri5",
        )

        x_s_hat, x_t_hat = ae.decode(
            traj_static[-1].cpu(),
            traj_temporal[-1].squeeze(1)[:, :31, :].cpu(),
            transform_and_unify_output=True,
            adjust_to_seq_lengths=seq_sizes,
        )
        return x_s_hat, x_t_hat, seq_sizes

In [ ]:
i = 0

while i < N_SAMPLES and GENERATE:
    try:
        with torch.no_grad():
            x_s_hat, x_t_hat, seq_sizes = generate_dataset(100)
        gen_df = ds.sample_to_df(x_s_hat.cpu(), x_t_hat.cpu(), seq_sizes)
        (synthetic_static_tensor, _, _) = ds.patient_dfs_to_tensors(
            ds.encode_features(
                gen_df.copy(),
                reassign_seq_lengths_weights=False,
                requires_median_delta_calc=False,
            )
        )
        assert synthetic_static_tensor.size(0) == encoded_static.size(0)
    except:
        continue

    static_gen_df = pd.concat(
        [df.iloc[0].to_frame().T for _, df in gen_df.groupby(by="REF")],
        ignore_index=True,
    )

    for feat in categorical_feats:
        if feat == ".":
            continue

        if feat not in categorical_feats_info:
            labels = list(map(lambda x: x.lower(), real[feat].unique()))

            if set(labels) == {"yes", "no", "unknown"}:
                label_order = ["yes", "no", "unknown"]
            elif set(labels) == {"male", "female", "unknown"}:
                label_order = ["male", "female", "unknown"]
            else:
                label_order = sorted(labels)

            categorical_feats_info[feat] = {"labels": label_order}
            categorical_feats_info[feat]["counts"] = np.zeros(
                (N_SAMPLES, len(label_order))
            )

        for j, label in enumerate(categorical_feats_info[feat]["labels"]):
            try:
                categorical_feats_info[feat]["counts"][i, j] = (
                    static_gen_df[feat].apply(lambda x: x.lower()).value_counts()[label]
                )
            except KeyError:
                categorical_feats_info[feat]["counts"][i, j] = 0

    gen_dfs.append(gen_df)
    i += 1
    print(f"{i}/{N_SAMPLES}", end="\r")

In [ ]:
fname = f"synthetic_dfs_n_samples={N_SAMPLES}_patientflow_{version}_seed={SEED}.pkl"

if GENERATE:
    with open(fname, "wb") as f:
        pickle.dump([gen_dfs, categorical_feats_info], f)
else:
    with open(fname, "rb") as f:
        gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
sns.set(font_scale=2.0)
sns.set_style("white")

In [ ]:
fig_categorical, axs_categorical = plt.subplot_mosaic(
    categorical_mosaic_grid, figsize=(25, 31), sharey=True
)

In [ ]:
static_real_df = pd.concat(
    [df.iloc[0].to_frame().T for _, df in real.groupby(by="REF")], ignore_index=True
)

In [ ]:
def plot_categorical_multiple_datasets():
    for feat in categorical_feats:
        if feat == ".":
            continue

        df = []
        for i, label in enumerate(categorical_feats_info[feat]["labels"]):
            df.append(
                pd.DataFrame(
                    {
                        feat.lower(): [label.lower()],
                        "type": ["real"],
                        "count": [
                            static_real_df[feat]
                            .apply(lambda x: x.lower())
                            .value_counts()[label]
                        ],
                    }
                )
            )
            for count in categorical_feats_info[feat]["counts"][:, i]:
                df.append(
                    pd.DataFrame(
                        {
                            feat.lower(): [label.lower()],
                            "type": ["fake"],
                            "count": [count],
                        }
                    )
                )
        df = pd.concat(df, ignore_index=True)

        label_order = categorical_feats_info[feat]["labels"]

        if "mutation" in feat.lower():
            uniques = [
                x
                for x in df[feat.lower()].unique()
                if x.lower() not in ("unknown", "no")
            ]
            label_mapping = {v: f"Mut. {i}" for i, v in enumerate(uniques, start=1)}
            df[feat.lower()] = df[feat.lower()].apply(
                lambda x: label_mapping[x] if x in label_mapping else x
            )
            print(f"MUTATION ENCODING FOR {feat}: {label_mapping}")
            label_order = sorted(label_mapping.values()) + ["no", "unknown"]

        if feat.lower() == "c9orf72":
            label_order = ["yes", "no", "unknown", "_"]

        g = sns.boxplot(
            data=df[df["type"] == "fake"],
            x=feat.lower(),
            y="count",
            ax=axs_categorical[feat],
            order=label_order,
        )
        for patch in g.patches:
            _r, _g, _b, _ = patch.get_facecolor()
            patch.set_facecolor((_r, _g, _b, 0.8))

        for _, row in df[df["type"] == "real"].iterrows():
            label_index = [xlabel.get_text() for xlabel in g.get_xticklabels()].index(
                row[feat.lower()]
            )
            label_xaxis = g.get_xticks()[label_index]
            g.hlines(
                y=row["count"],
                xmin=label_xaxis - 0.25,
                xmax=label_xaxis + 0.25,
                colors="red",
                zorder=10,
                linewidth=2,
            )
        g.set(ylabel="Number of\npatients")
        if feat in categorical_inverted_x_axis_feats:
            g.set_xticklabels(g.get_xticklabels(), rotation=45)

In [ ]:
def plot_categorical_single_dataset():
    for feat in categorical_feats:
        if feat == ".":
            continue

        df = []
        for i, label in enumerate(categorical_feats_info[feat]["labels"]):
            df.append(
                pd.DataFrame(
                    {
                        feat.lower(): [label.lower()],
                        "type": ["real"],
                        "count": [
                            static_real_df[feat]
                            .apply(lambda x: x.lower())
                            .value_counts()[label]
                        ],
                    }
                )
            )
            df.append(
                pd.DataFrame(
                    {
                        feat.lower(): [label.lower()],
                        "type": ["fake"],
                        "count": [categorical_feats_info[feat]["counts"][0, i]],
                    }
                )
            )

        df = pd.concat(df, ignore_index=True)
        label_order = categorical_feats_info[feat]["labels"]

        if "mutation" in feat.lower():
            uniques = [
                x
                for x in df[feat.lower()].unique()
                if x.lower() not in ("unknown", "no")
            ]
            label_mapping = {v: f"Mut. {i}" for i, v in enumerate(uniques, start=1)}
            df[feat.lower()] = df[feat.lower()].apply(
                lambda x: label_mapping[x] if x in label_mapping else x
            )
            print(f"MUTATION ENCODING FOR {feat}: {label_mapping}")
            label_order = sorted(label_mapping.values()) + ["no", "unknown"]

        if feat.lower() == "c9orf72":
            label_order = ["yes", "no", "unknown", "_"]

        g = sns.barplot(
            data=df,
            x=feat.lower(),
            y="count",
            hue="type",
            ax=axs_categorical[feat],
            order=label_order,
        )

In [ ]:
if N_SAMPLES > 1:
    plot_categorical_multiple_datasets()
else:
    plot_categorical_single_dataset()

In [ ]:
fig_categorical

In [ ]:
fig_categorical.tight_layout()
fig_categorical.savefig(f"categorical_comparison_patientflow_{version}.pdf")

In [ ]:
fig_numerical, axs_numerical = plt.subplot_mosaic(
    numerical_mosaic_grid, figsize=(20, 5)
)

In [ ]:
def plot_numerical_single_dataset():
    for feat in numerical_feats:
        df = pd.DataFrame(
            {
                feat.lower(): static_real_df[feat],
                "type": "real",
            }
        )
        df = pd.concat(
            [
                df,
                pd.DataFrame(
                    {
                        feat.lower(): list(
                            map(lambda x: x[1][feat].iloc[0], gen_dfs[0].groupby("REF"))
                        ),
                        "type": "fake",
                    }
                ),
            ],
            ignore_index=True,
        )
        g = sns.kdeplot(data=df, x=feat.lower(), hue="type", ax=axs_numerical[feat])

In [ ]:
def plot_numerical_multiple_datasets():
    fake_color = "steelblue"
    for feat in numerical_feats:
        feat_ax = axs_numerical[feat]
        df = pd.DataFrame({feat.lower(): static_real_df[feat]})
        sns.kdeplot(
            data=df, x=feat.lower(), ax=feat_ax, color="red", label="Real", zorder=10
        )

        for i in range(N_SAMPLES):
            df = pd.DataFrame(
                {
                    feat.lower(): list(
                        map(lambda x: x[1][feat].iloc[0], gen_dfs[i].groupby("REF"))
                    )
                }
            )
            sns.kdeplot(
                data=df, x=feat.lower(), ax=feat_ax, color=fake_color, alpha=0.1
            )

        feat_ax.plot([], [], color=fake_color, linestyle="solid", label="Synthetic")
        feat_ax.legend()

In [ ]:
if N_SAMPLES > 1:
    plot_numerical_multiple_datasets()
else:
    plot_numerical_single_dataset()

In [ ]:
fig_numerical

In [ ]:
fig_numerical.tight_layout()
fig_numerical.savefig(f"numerical_comparison_patientflow_{version}.pdf")

In [ ]:
def add_visit_num_col(x: pd.DataFrame) -> pd.DataFrame:
    x["visit_num"] = x.groupby("REF").cumcount() + 1
    return x

In [ ]:
def temporal_progression_comparison_single_dataset():
    real_for_line = real.copy()
    fake_for_line = gen_dfs[0].copy()

    real_for_line = real_for_line[["REF"] + [f"P{i}" for i in range(1, 13)]]
    fake_for_line = fake_for_line[["REF"] + [f"P{i}" for i in range(1, 13)]]

    real_for_line = real_for_line.groupby("REF", group_keys=False).apply(
        add_visit_num_col
    )
    fake_for_line = fake_for_line.groupby("REF", group_keys=False).apply(
        add_visit_num_col
    )
    real_for_line["type"] = "real"
    fake_for_line["type"] = "fake"
    line_df = pd.concat([real_for_line, fake_for_line], ignore_index=True)

    dfs = []

    for i, row in line_df.iterrows():
        for j in range(1, 13):
            dfs.append(
                pd.DataFrame(
                    {
                        "REF": row["REF"],
                        "type": {"real": "Real", "fake": "Synthetic"}[row["type"]],
                        "visit_num": row["visit_num"],
                        "question": [f"P{j}"],
                        "score": row[f"P{j}"],
                    }
                )
            )

    line_df = pd.concat(dfs, ignore_index=True)

    return (
        sns.FacetGrid(line_df, col_wrap=4, col="question", sharex=False)
        .map(sns.lineplot, "visit_num", "score", "type", estimator="mean")
        .add_legend()
        .set_xlabels("Visit Number")
    )

In [ ]:
def temporal_progression_comparison_multiple_datasets():
    _real_for_line = real.copy()
    fake_for_line = [gen_df.copy() for gen_df in gen_dfs]

    _real_for_line = _real_for_line[["REF"] + [f"P{i}" for i in range(1, 13)]]

    fake_for_line = [
        fake_for_line_df[["REF"] + [f"P{i}" for i in range(1, 13)]]
        for fake_for_line_df in fake_for_line
    ]

    _real_for_line = _real_for_line.groupby("REF", group_keys=False).apply(
        add_visit_num_col
    )
    real_for_line = []
    for i in range(1, _real_for_line["visit_num"].max() + 1):
        visit_df = _real_for_line[_real_for_line["visit_num"] == i]
        df_data = {"question": [], "score": [], "low": [], "high": [], "visit_num": []}
        for j in range(1, 13):
            mean = visit_df[f"P{j}"].mean()
            boots = bootstrap(visit_df[f"P{j}"].to_numpy(), func=np.mean)
            low, high = np.nanpercentile(boots, [2.5, 97.5])
            df_data["question"].append(j)
            df_data["score"].append(mean)
            df_data["low"].append(low)
            df_data["high"].append(high)
            df_data["visit_num"].append(i)
        real_for_line.append(pd.DataFrame(data=df_data))
    real_for_line = pd.concat(real_for_line, ignore_index=True)
    fake_for_line = [
        df.groupby("REF", group_keys=False).apply(add_visit_num_col)
        for df in fake_for_line
    ]

    agg_fake_for_line = []

    for df in fake_for_line:
        for i in range(1, df["visit_num"].max() + 1):
            visit_df = df[df["visit_num"] == i]
            df_data = {"question": [], "score": [], "std": [], "visit_num": []}
            for j in range(1, 13):
                df_data["question"].append(j)
                df_data["score"].append(visit_df[f"P{j}"].mean())
                df_data["std"].append(visit_df[f"P{j}"].std())
                df_data["visit_num"].append(i)
            agg_fake_for_line.append(pd.DataFrame(data=df_data))

    agg_fake_for_line = pd.concat(agg_fake_for_line, ignore_index=True)
    global_agg_fake_for_line = []

    for i in range(1, agg_fake_for_line["visit_num"].max() + 1):
        df_data = {"question": [], "score": [], "low": [], "high": [], "visit_num": []}
        visit_df = agg_fake_for_line[agg_fake_for_line["visit_num"] == i]
        for j in range(1, 13):
            visit_question_df = visit_df[visit_df["question"] == j]
            mean = visit_question_df["score"].mean()
            std = visit_question_df["std"].mean()
            df_data["question"].append(j)
            df_data["score"].append(mean)
            low, high = np.nanpercentile(visit_question_df["score"], [2.5, 97.5])
            df_data["low"].append(low)
            df_data["high"].append(high)
            df_data["visit_num"].append(i)
        global_agg_fake_for_line.append(pd.DataFrame(data=df_data))

    global_agg_fake_for_line = pd.concat(global_agg_fake_for_line, ignore_index=True)
    global_agg_fake_for_line["type"] = "Synthetic"

    real_for_line["type"] = "Real"
    line_df = pd.concat([real_for_line, global_agg_fake_for_line], ignore_index=True)
    g = (
        sns.FacetGrid(line_df, col_wrap=4, col="question", sharex=False)
        .map(sns.lineplot, "visit_num", "score", "type", estimator=None)
        .add_legend()
        .set_xlabels("Visit Number")
    )

    color_real = g.axes[0].get_lines()[0].get_color()
    color_fake = g.axes[0].get_lines()[1].get_color()

    for i in range(1, 13):
        g.axes[i - 1].fill_between(
            real_for_line[real_for_line["question"] == i]["visit_num"],
            real_for_line[real_for_line["question"] == i]["low"],
            real_for_line[real_for_line["question"] == i]["high"],
            color=color_real,
            alpha=0.2,
        )
        g.axes[i - 1].fill_between(
            global_agg_fake_for_line[global_agg_fake_for_line["question"] == i][
                "visit_num"
            ],
            global_agg_fake_for_line[global_agg_fake_for_line["question"] == i]["low"],
            global_agg_fake_for_line[global_agg_fake_for_line["question"] == i]["high"],
            color=color_fake,
            alpha=0.2,
        )

    return g

In [ ]:
if N_SAMPLES > 1:
    temp_g = temporal_progression_comparison_multiple_datasets()
else:
    temp_g = temporal_progression_comparison_single_dataset()

In [ ]:
temp_g.savefig(f"temporal_comparison_patientflow_{version}.pdf")

In [ ]:
# Correlations
feats = list(filter(lambda x: x != ".", numerical_feats + categorical_feats))

In [ ]:
import warnings

warnings.filterwarnings("error")

In [ ]:
def calculate_corrs(
    df: pd.DataFrame,
    feats: List[str],
    fake: bool = False,
    real_df: Optional[pd.DataFrame] = None,
):
    corrs = np.zeros((len(feats), len(feats)), dtype=np.float32)

    for col, feat1 in enumerate(feats):
        for row, feat2 in enumerate(feats):
            if feat1 == feat2:
                corrs[row, col] = 1.0

            elif feat1 in categorical_feats and feat2 in categorical_feats:
                obs = pd.crosstab(df[feat1], df[feat2])
                if fake:
                    obs = obs.reindex(real[feat1].unique(), fill_value=0)
                    obs = obs.reindex(real[feat2].unique(), axis=1, fill_value=0)
                    obs += 1
                corrs[row, col] = association(obs)

            elif feat1 in numerical_feats and feat2 in numerical_feats:
                corrs[row, col] = pearsonr(df[feat1], df[feat2])[0]

            else:
                corrs[row, col] = np.nan

    return corrs

In [ ]:
corrs_real = calculate_corrs(real, feats, fake=False)
parallel = Parallel(n_jobs=-1, verbose=10)
corrs_fake = parallel(
    delayed(calculate_corrs)(gen_df, feats, fake=True, real_df=real)
    for gen_df in gen_dfs
)

In [ ]:
corrs_fake = np.array(corrs_fake)
mean_corrs_fake = np.mean(corrs_fake, axis=0)

In [ ]:
mask = np.triu(np.ones_like(corrs_real, dtype=bool)) & ~np.triu(np.isnan(corrs_real))

In [ ]:
plt.rcdefaults()
global_font_size = 12

plt.rcParams["font.size"] = global_font_size  # Adjust the overall font size
plt.rcParams["axes.titlesize"] = global_font_size  # Title font size
plt.rcParams["axes.labelsize"] = global_font_size  # Label font size
plt.rcParams["xtick.labelsize"] = global_font_size  # X-axis tick label font size
plt.rcParams["ytick.labelsize"] = global_font_size  # Y-axis tick label font size

In [ ]:
fig_correlation_one, axs_correlation_one = plt.subplot_mosaic(
    [["Real", "Synthetic"]], figsize=(16.3, 7.5), sharey=True
)

In [ ]:
fig_correlation_two, axs_correlation_two = plt.subplots(figsize=(8, 8))

In [ ]:
cbar_kws = {"shrink": 1.0}

In [ ]:
sns.heatmap(
    corrs_real,
    xticklabels=feats,
    yticklabels=feats,
    vmin=-1.0,
    vmax=1.0,
    fmt=".2f",
    mask=mask,
    cmap=sns.diverging_palette(230, 20, as_cmap=True),
    linewidths=0.5,
    ax=axs_correlation_one["Real"],
    cbar_kws=cbar_kws,
)

In [ ]:
sns.heatmap(
    mean_corrs_fake,
    xticklabels=feats,
    yticklabels=feats,
    vmin=-1.0,
    vmax=1.0,
    fmt=".2f",
    mask=mask,
    cmap=sns.diverging_palette(230, 20, as_cmap=True),
    linewidths=0.5,
    ax=axs_correlation_one["Synthetic"],
    cbar_kws=cbar_kws,
)

In [ ]:
sns.heatmap(
    corrs_real - mean_corrs_fake,
    xticklabels=feats,
    yticklabels=feats,
    vmin=-2.0,
    vmax=2.0,
    fmt=".2f",
    mask=mask,
    cmap=sns.diverging_palette(230, 20, as_cmap=True),
    linewidths=0.5,
    ax=axs_correlation_two,
    cbar_kws=cbar_kws,
)

In [ ]:
frob_norm = np.linalg.norm(
    np.nan_to_num(corrs_real - mean_corrs_fake, copy=False, nan=0.0), ord="fro"
)
# axs_correlation_one["Real"].set_title("Real Data Correlation")
# axs_correlation_one["Synthetic"].set_title("Synthetic Data Correlation")
# axs_correlation_two.set_title(f"Correlation Difference (PCD: {frob_norm:.2f})")
print(frob_norm)

In [ ]:
fig_correlation_one

In [ ]:
fig_correlation_two

In [ ]:
mean_corrs_fake

In [ ]:
# frobenius norm
np.linalg.norm(
    np.nan_to_num(corrs_real - mean_corrs_fake, copy=False, nan=0.0), ord="fro"
)

In [ ]:
fig_correlation_one.savefig(
    f"correlation_comparison_patientflow_{version}.pdf", bbox_inches="tight"
)

In [ ]:
fig_correlation_two.savefig(
    f"correlation_comparison_patientflow_{version}_diff.pdf", bbox_inches="tight"
)